# Build & Push Parabricks Docker Image to ECR

This notebook builds the custom Parabricks image (extending `nvcr.io/nvidia/clara/clara-parabricks:4.5.1-1` with Databricks Container Services requirements) and pushes to AWS ECR.

**Prerequisites:**
- Must run on a **single-node cluster with Docker daemon access enabled** (Databricks Container Services)
- NGC API Key for pulling the base Parabricks image from `nvcr.io`
- AWS IAM permissions for ECR (create repository, push images)

**What the image includes:**
- NVIDIA Clara Parabricks 4.5.1-1 (GPU-accelerated genomics)
- JDK 8, Python 3.12, R (Databricks Container Services requirements)
- PySpark-compatible virtualenv at `/databricks/python3`
- Core data science libraries (pandas, numpy, pyarrow, matplotlib)

In [0]:
dbutils.widgets.text("ngc_api_key", "", "NGC API Key")
dbutils.widgets.text("ecr_repo_name", "genesis-workbench/parabricks-dbx", "ECR Repository Name")
dbutils.widgets.text("image_tag", "4.5.1-1-dbr15.4", "Image Tag")
dbutils.widgets.text("aws_region", "us-east-1", "AWS Region")

ngc_api_key = dbutils.widgets.get("ngc_api_key")
ecr_repo_name = dbutils.widgets.get("ecr_repo_name")
image_tag = dbutils.widgets.get("image_tag")
aws_region = dbutils.widgets.get("aws_region")

print(f"ECR Repo: {ecr_repo_name}")
print(f"Image Tag: {image_tag}")
print(f"AWS Region: {aws_region}")
print(f"NGC API Key: {'***' + ngc_api_key[-4:] if ngc_api_key else 'NOT SET'}")

if not ngc_api_key:
    raise ValueError("NGC API Key is required. Set the ngc_api_key widget.")

In [0]:
import subprocess
import sys

def run_cmd(cmd, description, check=True, capture=True):
    """Run a shell command and return the result."""
    print(f"\n{'='*60}")
    print(f"{description}")
    print(f"{'='*60}")
    result = subprocess.run(
        cmd, shell=True, capture_output=capture, text=True
    )
    if capture:
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(
            f"{description} failed (exit code {result.returncode}).\n"
            f"stderr: {result.stderr}\n\n"
            f"This notebook requires a Docker-enabled cluster.\n"
            f"Create a single-node cluster with 'Docker' enabled under\n"
            f"Advanced Options > Docker tab, or set docker_enabled=true\n"
            f"in your cluster policy."
        )
    return result

# Verify Docker daemon is accessible
run_cmd("docker --version", "Docker Version")
run_cmd("docker info --format '{{.ServerVersion}}'", "Docker Server Info")
print("\n Docker is available and running.")

In [0]:
# Login to NVIDIA NGC Container Registry
result = subprocess.run(
    ["docker", "login", "-u", "$oauthtoken", "-p", ngc_api_key, "nvcr.io"],
    capture_output=True, text=True
)

if result.returncode != 0:
    print(f"STDERR: {result.stderr}")
    raise RuntimeError(
        f"NGC authentication failed. Verify your API key is valid.\n"
        f"Generate a key at: https://catalog.ngc.nvidia.com/\n"
        f"Error: {result.stderr}"
    )

print("Successfully authenticated to nvcr.io (NVIDIA NGC)")
print(f"Base image: nvcr.io/nvidia/clara/clara-parabricks:4.5.1-1")

In [0]:
import boto3
from botocore.exceptions import ClientError

ecr_client = boto3.client("ecr", region_name=aws_region)

try:
    response = ecr_client.create_repository(
        repositoryName=ecr_repo_name,
        imageTagMutability="MUTABLE",
        imageScanningConfiguration={"scanOnPush": True},
    )
    repo_uri = response["repository"]["repositoryUri"]
    print(f"Created ECR repository: {repo_uri}")
except ClientError as e:
    if e.response["Error"]["Code"] == "RepositoryAlreadyExistsException":
        # Repository already exists, get its URI
        desc = ecr_client.describe_repositories(repositoryNames=[ecr_repo_name])
        repo_uri = desc["repositories"][0]["repositoryUri"]
        print(f"ECR repository already exists: {repo_uri}")
    else:
        raise

print(f"\nRepository URI: {repo_uri}")

In [0]:
import base64

# Get ECR authorization token
ecr = boto3.client("ecr", region_name=aws_region)
auth = ecr.get_authorization_token()
token = base64.b64decode(auth["authorizationData"][0]["authorizationToken"]).decode()
username, password = token.split(":")
registry = auth["authorizationData"][0]["proxyEndpoint"]

# Login Docker to ECR
result = subprocess.run(
    ["docker", "login", "-u", username, "-p", password, registry],
    capture_output=True, text=True
)

if result.returncode != 0:
    raise RuntimeError(f"ECR Docker login failed: {result.stderr}")

ecr_uri = auth["authorizationData"][0]["proxyEndpoint"].replace("https://", "")
print(f"Authenticated to ECR: {ecr_uri}")
print(f"Full image target: {ecr_uri}/{ecr_repo_name}:{image_tag}")

In [0]:
import time

# Paths to Dockerfile and build context
dockerfile_path = "/Workspace/Users/andrew_forman@eisai.com/genesis-workbench/modules/parabricks/docker/dockerfile"
docker_context_path = "/Workspace/Users/andrew_forman@eisai.com/genesis-workbench/modules/parabricks/docker/"
full_image_uri = f"{ecr_uri}/{ecr_repo_name}:{image_tag}"

print(f"Building image: {full_image_uri}")
print(f"Dockerfile: {dockerfile_path}")
print(f"Context: {docker_context_path}")
print(f"\nThis may take 10-20 minutes for the initial build...\n")
print("=" * 60)

start_time = time.time()

# Build using docker buildx for linux/amd64
build_cmd = [
    "docker", "buildx", "build",
    "--platform", "linux/amd64",
    "-t", full_image_uri,
    "-f", dockerfile_path,
    docker_context_path
]

proc = subprocess.Popen(
    build_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Stream output in real-time
for line in proc.stdout:
    print(line, end="")

proc.wait()

elapsed = time.time() - start_time

if proc.returncode != 0:
    raise RuntimeError(
        f"Docker build failed with exit code {proc.returncode}.\n"
        f"Review the output above for errors."
    )

print(f"\n{'='*60}")
print(f"Build completed successfully in {elapsed/60:.1f} minutes")
print(f"Image: {full_image_uri}")

In [0]:
print(f"Pushing image: {full_image_uri}")
print(f"This may take several minutes depending on image size...\n")
print("=" * 60)

start_time = time.time()

push_proc = subprocess.Popen(
    ["docker", "push", full_image_uri],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in push_proc.stdout:
    print(line, end="")

push_proc.wait()

elapsed = time.time() - start_time

if push_proc.returncode != 0:
    raise RuntimeError(
        f"Docker push failed with exit code {push_proc.returncode}.\n"
        f"Verify ECR permissions and repository exists."
    )

print(f"\n{'='*60}")
print(f"Push completed successfully in {elapsed/60:.1f} minutes")
print(f"Image available at: {full_image_uri}")

In [0]:
print(f"""
{'='*60}
=== Parabricks Docker Image Ready ===
{'='*60}

Image URI: {full_image_uri}

Add to genesis-workbench/modules/parabricks/module.env:
  parabricks_docker_image={full_image_uri}
  parabricks_docker_userid=
  parabricks_docker_token=

Note: ECR images on AWS don't need basic_auth - the cluster's
instance profile handles authentication. Leave userid/token empty.

Next: Run deploy.sh from the parabricks module directory:
  cd genesis-workbench/modules/parabricks
  ./deploy.sh aws

{'='*60}
""")

In [0]:
import json

# Inspect the built image
result = subprocess.run(
    ["docker", "inspect", full_image_uri],
    capture_output=True, text=True
)

if result.returncode != 0:
    print(f"Warning: Could not inspect image: {result.stderr}")
else:
    inspect_data = json.loads(result.stdout)
    if inspect_data:
        img = inspect_data[0]
        size_mb = img.get("Size", 0) / (1024 * 1024)
        created = img.get("Created", "unknown")
        arch = img.get("Architecture", "unknown")
        os_name = img.get("Os", "unknown")
        
        print(f"Image: {full_image_uri}")
        print(f"Created: {created}")
        print(f"Size: {size_mb:.0f} MB")
        print(f"Architecture: {arch}")
        print(f"OS: {os_name}")
        
        # Print key environment variables
        env_vars = img.get("Config", {}).get("Env", [])
        print(f"\nKey Environment Variables:")
        for env in env_vars:
            if any(k in env for k in ["PYSPARK", "CUDA", "NVIDIA", "PATH", "JAVA"]):
                print(f"  {env}")
        
        print(f"\n Image verified and ready for deployment.")